#### Outcome Model

##### Importing Libraries and Loading Data

In [1]:
# Libraries, and every application with its category (from the form or the classifier)

import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from planscope.evaluate import evaluate_risk
from planscope.outcome_data import CATEGORIES_PATH, CLEAN_PATH, OUT_PATH, strip_post_submission

In [2]:
apps = pd.read_parquet(CLEAN_PATH).drop(columns='category')

In [3]:
apps.head()

,id,lpa_name,lpa_app_no,valid_date,decision_date,outcome,decision_process,application_type,description,units_proposed,units_existing,storeys,site_area_ha,lon,lat,uprn,url_planning_app
0,Hounslow-00249_R_O_44_PA1(4-6),Hounslow,00249/R/O/44/PA1(4-6),2022-03-29,2022-07-22,APPROVED,NaN,All Other,Details of Condition 4 (Waste and Recycle) Con...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Hounslow-01122_22_P1_,Hounslow,01122/22/P1,2019-10-03,2019-10-18,OTHER,NaN,All Other,SIDE EXTENSION & RE STUDIO,NaN,NaN,NaN,NaN,-0.303379,51.488912,NaN,NaN
2,Islington-P2025_0550_TRE,Islington,P2025/0550/TRE,2025-02-26,NaT,NaN,NaN,All Other,"Work to be carried out at St Johns Square, Cle...",NaN,NaN,NaN,NaN,-0.103124,51.523063,5300085709,NaN
3,Islington-P2025_0132_LBC,Islington,P2025/0132/LBC,2025-01-27,NaT,NaN,NaN,All Other,Internal adaptation of a space at lower ground...,NaN,NaN,NaN,NaN,-0.105958,51.522498,10093625150,NaN
4,Islington-P2025_0161_LBC,Islington,P2025/0161/LBC,2025-01-17,NaT,APPROVED,NaN,All Other,Proposal to remove and infill non original sin...,NaN,NaN,NaN,NaN,-0.106242,51.538239,5300035278,NaN


In [4]:
apps = apps.merge(pd.read_parquet(CATEGORIES_PATH, columns=['id', 'category']), on='id')

In [5]:
apps.head()

,id,lpa_name,lpa_app_no,valid_date,decision_date,outcome,decision_process,application_type,description,units_proposed,units_existing,storeys,site_area_ha,lon,lat,uprn,url_planning_app,category
0,Hounslow-00249_R_O_44_PA1(4-6),Hounslow,00249/R/O/44/PA1(4-6),2022-03-29,2022-07-22,APPROVED,NaN,All Other,Details of Condition 4 (Waste and Recycle) Con...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,conditions
1,Hounslow-01122_22_P1_,Hounslow,01122/22/P1,2019-10-03,2019-10-18,OTHER,NaN,All Other,SIDE EXTENSION & RE STUDIO,NaN,NaN,NaN,NaN,-0.303379,51.488912,NaN,NaN,householder
2,Islington-P2025_0550_TRE,Islington,P2025/0550/TRE,2025-02-26,NaT,NaN,NaN,All Other,"Work to be carried out at St Johns Square, Cle...",NaN,NaN,NaN,NaN,-0.103124,51.523063,5300085709,NaN,trees
3,Islington-P2025_0132_LBC,Islington,P2025/0132/LBC,2025-01-27,NaT,NaN,NaN,All Other,Internal adaptation of a space at lower ground...,NaN,NaN,NaN,NaN,-0.105958,51.522498,10093625150,NaN,listed_building
4,Islington-P2025_0161_LBC,Islington,P2025/0161/LBC,2025-01-17,NaT,APPROVED,NaN,All Other,Proposal to remove and infill non original sin...,NaN,NaN,NaN,NaN,-0.106242,51.538239,5300035278,NaN,listed_building


In [6]:
decided = apps[apps['outcome'].isin(['APPROVED', 'REFUSED'])]

In [7]:
decided.head()

,id,lpa_name,lpa_app_no,valid_date,decision_date,outcome,decision_process,application_type,description,units_proposed,units_existing,storeys,site_area_ha,lon,lat,uprn,url_planning_app,category
0,Hounslow-00249_R_O_44_PA1(4-6),Hounslow,00249/R/O/44/PA1(4-6),2022-03-29,2022-07-22,APPROVED,NaN,All Other,Details of Condition 4 (Waste and Recycle) Con...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,conditions
4,Islington-P2025_0161_LBC,Islington,P2025/0161/LBC,2025-01-17,NaT,APPROVED,NaN,All Other,Proposal to remove and infill non original sin...,NaN,NaN,NaN,NaN,-0.106242,51.538239,5300035278,NaN,listed_building
6,Hillingdon-18974_APP_2021_4641,Hillingdon,18974/APP/2021/4641,2021-12-29,2022-02-16,APPROVED,Delegated,All Other,Conversion of roofspace to habitable use to in...,0.0,0.0,NaN,0.039,-0.438001,51.544655,NaN,http://planning.hillingdon.gov.uk/OcellaWeb/pl...,lawful_development
7,Camden-2022_5042_P,Camden,2022/5042/P,2022-11-17,2023-12-11,REFUSED,Delegated,Householder,Erection of a roof extension for the use of th...,NaN,NaN,NaN,0.027,-0.203063,51.553577,5027372,https://planningrecords.camden.gov.uk/Northgat...,householder
8,Westminster-19_09787_FULL,Westminster,19/09787/FULL,2019-12-16,2020-02-06,REFUSED,Delegated,All Other,"Installation of full openable shopfront, insta...",NaN,NaN,NaN,0.059,-0.132071,51.513510,010033548067,NaN,full_planning


In [8]:
apps.shape

(899237, 18)

In [9]:
decided.shape

(721849, 18)

##### Audit

In [10]:
# Share of applications with a decision, by quarter submitted, recent ones are not decided yet

apps['outcome'].isin(['APPROVED', 'REFUSED']).groupby(apps['valid_date'].dt.to_period('Q')).mean().tail(10).round(3)

valid_date
2024Q3    0.827
2024Q4    0.823
2025Q1    0.839
2025Q2    0.815
2025Q3    0.824
2025Q4    0.815
2026Q1    0.805
2026Q2    0.778
2026Q3    0.593
2026Q4    0.080
Freq: Q-DEC, Name: outcome, dtype: float64

In [11]:
# Refusal rate by year submitted. It drifts, so the test set must come after the training set

decided['outcome'].eq('REFUSED').groupby(decided['valid_date'].dt.year).mean().round(3)

valid_date
2019    0.185
2020    0.205
2021    0.198
2022    0.188
2023    0.191
2024    0.175
2025    0.157
2026    0.149
Name: outcome, dtype: float64

In [12]:
# Barking & Dagenham against the rest of London, by type: share decided, and refusal rate when decided

area = apps['lpa_name'].eq('Barking & Dagenham').map({True: 'Barking & Dagenham', False: 'rest of London'})
decided_share = pd.crosstab(apps['category'], area, values=apps['outcome'].isin(['APPROVED', 'REFUSED']), aggfunc='mean')
refusal_rate = pd.crosstab(decided['category'], area.loc[decided.index], values=decided['outcome'].eq('REFUSED'), aggfunc='mean')
pd.concat({'decided share': decided_share, 'refusal rate': refusal_rate}, axis=1).round(3)

decided share                      refusal rate  \
lpa_name           Barking & Dagenham rest of London Barking & Dagenham   
category                                                                  
advert                          0.000          0.881                NaN   
amendment                       0.284          0.859              0.040   
conditions                      0.054          0.813              0.000   
full_planning                   0.908          0.827              0.408   
householder                     0.939          0.914              0.328   
lawful_development              0.022          0.875              0.733   
listed_building                 0.167          0.826              0.000   
other                             NaN          0.829                NaN   
prior_approval                  0.013          0.579              0.458   
trees                           0.000          0.494                NaN   

                                   
lpa_name           rest of London  
category                           
advert                      0.137  
amendment                   0.148  
conditions                  0.086  
full_planning               0.236  
householder                 0.180  
lawful_development          0.182  
listed_building             0.074  
other                       0.088  
prior_approval              0.487  
trees                       0.089

In [13]:
# Text added after submission: appeal links only exist after a refusal; amendment notes appear during negotiation

text = decided['description'].dropna()
note = np.select([text.str.contains('linked cases:', case=False, regex=False), strip_post_submission(text) != text],
                 ['appeal link', 'amendment note'], 'none')
decided.loc[text.index, 'outcome'].eq('REFUSED').groupby(note).agg(['size', 'mean']).round(3)

,size,mean
amendment note,11430,0.152
appeal link,49,1.000
none,710262,0.184


##### Baselines

In [14]:
# The modelling table from outcome_data.py, split by date, first baseline: everyone gets the training refusal rate

df = pd.read_parquet(OUT_PATH)
train, val, test = [df[df['split'] == name] for name in ['train', 'val', 'test']]
results = {'constant': evaluate_risk(test['refused'], np.full(len(test), train['refused'].mean()))}

In [15]:
# Baseline: each authority's refusal rate for the type, shrunk toward London's rate for that type when cases are few

M = 20  # pseudo-cases at London's rate added to every authority and type
type_rate = train.groupby('category')['refused'].mean()
counts = train.groupby(['lpa_name', 'category'])['refused'].agg(['sum', 'size'])
prior = type_rate.reindex(counts.index.get_level_values('category')).to_numpy()
group_rate = ((counts['sum'] + M * prior) / (counts['size'] + M)).rename('rate')
risk = test.join(group_rate, on=['lpa_name', 'category'])['rate'].fillna(test['category'].map(type_rate))
results['authority x type'] = evaluate_risk(test['refused'], risk.fillna(train['refused'].mean()))

In [16]:
# Text only: TF-IDF words and word pairs + logistic regression

text_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True), LogisticRegression(C=0.5, max_iter=1000))
text_model.fit(train['text'], train['refused'])
results['text only'] = evaluate_risk(test['refused'], text_model.predict_proba(test['text'])[:, 1])

##### Text + Authority + Type + Form

In [17]:
# One model: text features plus one-hot authority, type, authority x type and binned form numbers

GROUPS = ['lpa_name', 'category', 'lpa_category', 'units_proposed', 'storeys', 'site_area_ha']


def outcome_model(C, groups=GROUPS):
    features = make_column_transformer(
        (TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True), 'text'),
        (OneHotEncoder(handle_unknown='ignore', min_frequency=20), groups),
    )
    return make_pipeline(features, LogisticRegression(C=C, max_iter=2000))


# Choose C by validation PR-AUC
best_score, model = 0, None
for C in [0.25, 0.5, 1]:
    candidate = outcome_model(C).fit(train, train['refused'])
    score = average_precision_score(val['refused'], candidate.predict_proba(val)[:, 1])
    print(f'C={C}: validation PR-AUC {score:.4f}')
    if score > best_score:
        best_score, model = score, candidate

C=0.25: validation PR-AUC 0.4377
C=0.5: validation PR-AUC 0.4400
C=1: validation PR-AUC 0.4400


In [18]:
# Does the application form help? The same model without the three form numbers

no_form = outcome_model(model[-1].C, GROUPS[:3]).fit(train, train['refused'])
results['text + groups, no form'] = evaluate_risk(test['refused'], no_form.predict_proba(test)[:, 1])

In [19]:
# Every model on the test year, with 95% confidence intervals

results['text + groups'] = evaluate_risk(test['refused'], model.predict_proba(test)[:, 1])
pd.concat(results)

score  ci_low  ci_high
constant               roc_auc  0.5000  0.5000   0.5000
                       pr_auc   0.1582  0.1559   0.1604
                       brier    0.1344  0.1330   0.1358
authority x type       roc_auc  0.6878  0.6832   0.6927
                       pr_auc   0.3050  0.2981   0.3123
                       brier    0.1266  0.1251   0.1280
text only              roc_auc  0.7517  0.7473   0.7562
                       pr_auc   0.3866  0.3785   0.3946
                       brier    0.1171  0.1156   0.1184
text + groups, no form roc_auc  0.7600  0.7559   0.7645
                       pr_auc   0.4043  0.3963   0.4125
                       brier    0.1163  0.1148   0.1177
text + groups          roc_auc  0.7449  0.7405   0.7493
                       pr_auc   0.3917  0.3835   0.4000
                       brier    0.1212  0.1198   0.1226